In [24]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
import torch
import os
import librosa
import whisper
from google.colab import drive
import spacy
import textstat
from collections import Counter
drive.mount('/content/drive')
import torch

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 7.0 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=223f624a3e743cb5a7fd719826a053733e26a26bbe2b11be13e8dbff236154af
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [13]:
model = whisper.load_model('medium')

100%|█████████████████████████████████████| 1.42G/1.42G [00:41<00:00, 36.4MiB/s]


In [ ]:
data = model.transcribe("my_audio.aac")

In [ ]:
text_data = data['text']

In [14]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.8 MB/s eta 0:00:00


In [29]:
def convert_audio_to_text(audio_path):
    model = whisper.load_model('medium')
    result = model.transcribe(audio_path)
    return result['text']

nlp = spacy.load("en_core_web_sm")
filler_words = ["umm", "uh", "uhh", "like", "you know", "basically", "so", "actually"]

def extract_text_features(text):
    doc = nlp(text)

    # 1. Filler count
    tokens = [t.text.lower() for t in doc]
    filler_count = sum(tokens.count(w) for w in filler_words)

    # 2. Repetition
    word_freq = Counter(tokens)
    repetition_score = sum([freq for word, freq in word_freq.items() if freq > 2])

    # 3. Grammar errors (spaCy-based)
    grammar_errors = sum([1 for token in doc if token.dep_ == "dep"])

    # 4. Sentence length stats
    sentence_lengths = [len(sent.text.split()) for sent in doc.sents]
    avg_sentence_length = sum(sentence_lengths) / len(sentence_lengths)

    # 5. Readability
    readability = textstat.flesch_reading_ease(text)

    return {
        "filler_count": filler_count,
        "repetition_score": repetition_score,
        "grammar_errors": grammar_errors,
        "avg_sentence_length": avg_sentence_length,
        "readability": readability
    }

# text = "Umm I think maybe this is like not correct uh but you know I will try."
# features = extract_text_features(text_data)
# print(features)


Model Training

In [ ]:
# Train  My Model
# Disable W&B to avoid login errors
os.environ["WANDB_DISABLED"] = "true"

# Load dataset
df = pd.read_csv("cleaned.csv")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize
encodings = tokenizer(
    list(df["text"]),
    truncation=True,
    padding=True,
    return_tensors="pt"
)

# Torch dataset
class ConfidenceDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.float)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]  # shape: scalar float
        return item

    def __len__(self):
        return len(self.labels)

dataset = ConfidenceDataset(encodings, df["score"])

# Model for regression (1 output)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=1  # regression
)

# Training settings
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    logging_steps=10,
    eval_strategy="no",
    report_to="none",          # prevents wandb/tensorboard
    save_strategy="no",        # faster training
    logging_dir="./logs"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# Train
trainer.train()

# Save model
trainer.save_model("confidence_model")
tokenizer.save_pretrained("confidence_model")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
10,0.076500
20,0.032100
30,0.046800
40,0.046200
50,0.034900
60,0.028900
70,0.024900
80,0.024300
90,0.032800
100,0.022000


('confidence_model/tokenizer_config.json',
 'confidence_model/special_tokens_map.json',
 'confidence_model/vocab.txt',
 'confidence_model/added_tokens.json',
 'confidence_model/tokenizer.json')

In [ ]:
trainer.save_model("/content/drive/MyDrive/confidence_model")
tokenizer.save_pretrained("/content/drive/MyDrive/confidence_model")

('/content/drive/MyDrive/confidence_model/tokenizer_config.json',
 '/content/drive/MyDrive/confidence_model/special_tokens_map.json',
 '/content/drive/MyDrive/confidence_model/vocab.txt',
 '/content/drive/MyDrive/confidence_model/added_tokens.json',
 '/content/drive/MyDrive/confidence_model/tokenizer.json')

Loading My Model and Evaluation Then

In [16]:
# Load model + tokenizer
model_path = "/content/drive/MyDrive/confidence_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()   # evaluation mode

def get_confidence_score(text):
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)

    # Regression output is in logits
    score = outputs.logits.item()

    # Clamp score between 0–1
    score = max(0.0, min(1.0, score))

    return score
# -------------------------
# Example
# -------------------------
# text = "hello"
# score = get_confidence_score(text)

# print("Confidence Score:", score)

In [21]:
print(get_confidence_score("Hello this me Bydiii.I am very good in programming.umm i think i will do this.Lets talk more about this role"))
print(get_confidence_score("Definatly I will do this task effectively."))
# print(get_confidence_score("I will do this task easily for you and i  fit for this role."))
# print(get_confidence_score("I am fully confident in my ability to handle this task and deliver strong results."))


0.39053449034690857
0.8820945620536804


In [ ]:
calculate_confidence(features)

0.747

Get Audios Features like Energy ,Pitch,Speech rate etc

In [22]:
import librosa
def extract_audio_features(audio_path):
    y, sr = librosa.load(audio_path)

    features = {}

    # 1. Pitch (fundamental frequency)
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch = np.mean(pitches[pitches > 0])
    features["pitch"] = pitch

    # 2. Volume / Energy
    energy = np.mean(y**2)
    features["energy"] = energy

    # 3. Speaking Rate
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features["speech_rate"] = tempo

    # 4. MFCCs (best for ML training)
    mfcc = librosa.feature.mfcc(y=y, sr=sr)
    features["mfcc_mean"] = np.mean(mfcc)

    return features

In [25]:
audio_features=extract_audio_features('my_audio.aac')

/tmp/ipython-input-285842431.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Calculate Confidence Score on base of Text Features ... Like Grammar , Repetition score, etc

In [26]:
def calculate_confidence(text_features, audio_features):

    # -----------------------------
    # TEXT FEATURES
    # -----------------------------
    filler = text_features['filler_count']
    repetition = text_features['repetition_score']
    grammar = text_features['grammar_errors']
    length = text_features['avg_sentence_length']
    readability = text_features['readability']

    filler_score = max(0, 1 - filler/10)
    repetition_score = max(0, 1 - repetition/100)
    grammar_score = max(0, 1 - grammar/10)

    # Ideal sentence length = 10–30 words
    if length < 10:
        length_score = length/10
    elif 10 <= length <= 30:
        length_score = 1.0
    else:
        length_score = max(0, 1 - ((length - 30)/40))

    readability_score = min(readability/50, 1.0)

    text_confidence = (
        0.22 * filler_score +
        0.20 * repetition_score +
        0.18 * grammar_score +
        0.15 * length_score +
        0.25 * readability_score
    )

    # -----------------------------
    # AUDIO FEATURES
    # -----------------------------
    pitch = float(audio_features['pitch'])
    energy = float(audio_features['energy'])
    speech_rate = float(audio_features['speech_rate'])
    mfcc_mean = float(audio_features['mfcc_mean'])

    # Pitch (ideal range 130–220 Hz)
    if pitch < 100:
        pitch_score = pitch / 100
    elif 100 <= pitch <= 220:
        pitch_score = 1.0
    else:
        pitch_score = max(0, 1 - ((pitch - 220)/200))

    # Energy (normalized 0–1, more energy = more confidence)
    energy_score = min(energy * 5000, 1.0)

    # Speech rate (ideal 110–160 words/min)
    if speech_rate < 80:
        speech_rate_score = speech_rate / 80
    elif 80 <= speech_rate <= 160:
        speech_rate_score = 1.0
    else:
        speech_rate_score = max(0, 1 - ((speech_rate - 160)/100))

    # MFCC mean (closer to 0 = stronger voice)
    mfcc_score = max(0, 1 - abs(mfcc_mean) / 40)

    audio_confidence = (
        0.30 * pitch_score +
        0.25 * energy_score +
        0.25 * speech_rate_score +
        0.20 * mfcc_score
    )

    # -----------------------------
    # FINAL HYBRID CONFIDENCE SCORE
    # -----------------------------
    final_confidence = (0.55 * text_confidence) + (0.45 * audio_confidence)
    return round(final_confidence, 3)

In [ ]:
calculate_confidence(features,audio_features)

/tmp/ipython-input-1653619545.py:41: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  speech_rate = float(audio_features['speech_rate'])


0.69

In [27]:
def confidence_score(audio_path):
    text = convert_audio_to_text(audio_path)
    text_score = get_confidence_score(text)
    text_features = extract_text_features(audio_path)
    audio_features = extract_audio_features(audio_path)
    confidence = calculate_confidence(text_features, audio_features)
    score = (text_score+confidence) / 2
    return score

In [35]:
# audio_score = confidence_score('my_audio.aac')
audio_score = confidence_score('my_audio.aac')

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/tmp/ipython-input-285842431.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipython-input-3632967286.py:39: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  speech_rate = float(audio_features['speech_rate'])


In [34]:
print(audio_score)

0.5331947908401489


In [ ]:
import sounddevice as sd
import soundfile as sf
import threading

# Settings
samplerate = 16000
channels = 1
filename = "live_recording.wav"

recording = []
is_recording = False

def record_audio():
    global recording, is_recording
    is_recording = True
    print("🎙️ Recording... Press ENTER to stop.\n")

    def callback(indata, frames, time, status):
        if is_recording:
            recording.append(indata.copy())

    with sd.InputStream(samplerate=samplerate, channels=channels, callback=callback):
        while is_recording:
            sd.sleep(100)

def stop_recording():
    global is_recording
    input()  # wait for ENTER
    is_recording = False
    print("🛑 Stopped Recording.")

# Start threads
t1 = threading.Thread(target=record_audio)
t2 = threading.Thread(target=stop_recording)

t1.start()
t2.start()

t1.join()

# Save
audio_np = b"".join([d.tobytes() for d in recording])
audio_np = b"".join(recording)  # alternative

full_audio = b''.join([d.tobytes() for d in recording])

# Convert back to float32 PCM
import numpy as np
audio_array = np.frombuffer(full_audio, dtype=np.float32).reshape(-1, 1)

sf.write(filename, audio_array, samplerate)

print(f"🎧 Saved as {filename}")


OSError: PortAudio library not found

In [5]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 60.8 MB/s eta 0:00:00


In [3]:
%%writefile app.py

import streamlit as st

st.set_page_config(page_title="Colab Streamlit App")

st.title("Streamlit Running on Google Colab")
st.write("This app is served from Google Colab using ngrok.")

name = st.text_input("Enter your name")
if name:
    st.success(f"Hello, {name}!")

Overwriting app.py


In [50]:
from pyngrok import ngrok
ngrok.set_auth_token("36yZxMAY1z20UptgvPlZok5SJmI_M8H8JNYqk6MWvuW4bjkX")

In [6]:
!streamlit run app.py &>/content/streamlit.log &
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501


/bin/bash: line 1: ./cloudflared-linux-amd64: No such file or directory
